In [0]:
%sql
use catalog workspace;
use schema default;

In [0]:
from pyspark.sql import Row
from datetime import datetime, timedelta
import random

# Build a small batch of event rows inline

base_time = datetime(2024,1,15,9,0,0)
batch_1 = [
Row(event_id=i,
    event_type = random.choice(["click", "view", "purchase"]),
    user_id=random.randint(100,199),
    event_ts=base_time + timedelta(seconds= i*30))
    for i in range(1,11)
]

df_batch_1 = spark.createDataFrame(batch_1)
display(df_batch_1)


In [0]:
df_batch_1.write \
  .format("delta") \
  .mode("append") \
  .saveAsTable("workspace.default.events")
print("Batch 1 written -  10 rows appended")

In [0]:
base_time = datetime(2024,1,15,9,0,0)
batch_2 = [
Row(event_id=i,
    event_type = random.choice(["click", "view", "purchase"]),
    user_id=random.randint(100,199),
    event_ts=base_time + timedelta(seconds= i*30))
    for i in range(1,11)
]

df_batch_2 = spark.createDataFrame(batch_1)
display(df_batch_2)
df_batch_2.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.default.events")
print("Batch 2 written -  10 rows appended")

In [0]:
%sql
describe history events

In [0]:
%sql
select * from events;

#### Reading Delta Tables from External Tables

In [0]:
location = spark.sql("DESCRIBE DETAIL events").select("location").collect()[0][0]
print(f"Table events is stored at {location}")

In [0]:
%sql
DESCRIBE DETAIL default.events;

In [0]:
df_path=spark.read.format("delta").load(location)
print(f"Reading from: {location}")
print(f"Row count: {df_path.count()}")
display(df_path)

Batch Read - Catalog Based

In [0]:
df_catalog = spark.table("events")
print(f"Row Count: {df_catalog.count()}")
display(df_catalog)